<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Assignment_Linear_Regression_and_Model_Interpretation_Subjective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA = """vehicle_age,km_driven,mileage,engine,max_power,selling_price,fuel_Diesel,fuel_Petrol,trans_Manual,trans_Auto
3,45000,17.5,1248,82.0,550000,1,0,1,0
5,80000,19.2,1497,103.0,400000,0,1,1,0
1,12000,14.8,1969,140.0,900000,1,0,0,1
7,110000,21.3,1197,68.0,280000,0,1,1,0
2,30000,15.0,1984,190.0,1500000,1,0,0,1
8,150000,22.0,998,67.0,220000,0,1,1,0
4,60000,18.5,1461,98.0,480000,1,0,1,0
6,95000,20.1,1248,74.0,320000,0,1,1,0
2,22000,13.0,2179,210.0,2200000,1,0,0,1
9,170000,23.5,796,46.0,180000,0,1,1,0
3,40000,16.8,1497,108.0,620000,0,1,0,1
1,8000,12.5,2993,258.0,3800000,1,0,0,1
"""

def build_pipeline(data_str):
    # Step 1: Load from inlined CSV string
    df = pd.read_csv(StringIO(data_str))

    # Step 2: Separate X and y
    X = df.drop(columns=["selling_price"])
    y = df["selling_price"]

    # Step 3: Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Step 4: Scale numerical columns (fit on train only)
    numerical_cols = [
        "vehicle_age",
        "km_driven",
        "mileage",
        "engine",
        "max_power"
    ]

    scaler = StandardScaler()

    X_train = X_train.copy()
    X_test = X_test.copy()

    X_train[numerical_cols] = scaler.fit_transform(
        X_train[numerical_cols]
    )

    X_test[numerical_cols] = scaler.transform(
        X_test[numerical_cols]
    )

    # Step 5: Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Step 6: Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Step 7: Compute metrics
    def regression_metrics(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        return mae, mse, rmse, r2

    train_mae, train_mse, train_rmse, train_r2 = regression_metrics(
        y_train, y_train_pred
    )

    test_mae, test_mse, test_rmse, test_r2 = regression_metrics(
        y_test, y_test_pred
    )

    print("=== Train Metrics ===")
    print(f"MAE : {train_mae:,.2f}")
    print(f"MSE : {train_mse:,.2f}")
    print(f"RMSE: {train_rmse:,.2f}")
    print(f"R²  : {train_r2:.4f}")

    print("\n=== Test Metrics ===")
    print(f"MAE : {test_mae:,.2f}")
    print(f"MSE : {test_mse:,.2f}")
    print(f"RMSE: {test_rmse:,.2f}")
    print(f"R²  : {test_r2:.4f}")

    # Step 8: Top-3 features by absolute coefficient value
    coef_df = pd.DataFrame({
        "feature": X_train.columns,
        "coefficient": model.coef_,
        "abs_coefficient": np.abs(model.coef_)
    })

    top3 = (
        coef_df
        .sort_values("abs_coefficient", ascending=False)
        .head(3)
    )

    print("\n=== Top-3 Features by |Coefficient| ===")
    print(top3[["feature", "coefficient"]].to_string(index=False))

    return {
        "model": model,
        "scaler": scaler,
        "top3_features": top3,
        "train_metrics": {
            "MAE": train_mae,
            "MSE": train_mse,
            "RMSE": train_rmse,
            "R2": train_r2
        },
        "test_metrics": {
            "MAE": test_mae,
            "MSE": test_mse,
            "RMSE": test_rmse,
            "R2": test_r2
        }
    }

if __name__ == "__main__":
    build_pipeline(DATA)

=== Train Metrics ===
MAE : 42,334.39
MSE : 3,416,816,358.02
RMSE: 58,453.54
R²  : 0.9974

=== Test Metrics ===
MAE : 585,528.46
MSE : 423,265,242,601.87
RMSE: 650,588.38
R²  : -10.3577

=== Top-3 Features by |Coefficient| ===
    feature   coefficient
     engine  1.332360e+06
vehicle_age  1.150637e+06
    mileage -4.804693e+05
